<a href="https://colab.research.google.com/github/renato2007-hub/online-retail-ml/blob/main/Proyecto_Final_Aprendizaje_Estad%C3%ADstico_Renato_P%C3%A9rez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto de Aprendizaje Estadístico
## Análisis Online Retail II UCI

Este notebook incluye:
1. Fase 1: Formulación del Problema y Entendimiento de los Datos
2. Fase 2: Preparación y Exploración de Datos (EDA)
3. Fase 3: Modelado y Evaluación
4. Fase 4: Interpretación y Comunicación de Resultados


# 1. Fase I: Formulación del Problema y Entendimiento de los Datos

Descripción: Transacciones de una tienda minorista online del Reino Unido entre 01/12/2009 y 09/12/2011. Contiene información sobre facturas, productos, cantidades, precios, clientes y países.
Posibles Problemas a Abordar: Segmentación de clientes (RFM, clustering), predicción de próxima compra, detección de productos frecuentemente comprados juntos, predicción de valor de vida del cliente (CLV) simplificado.
Enlace (UCI): https://archive.ics.uci.edu/dataset/502/online+retail+ii. Descargué el archivo excel a mi computadora y de ahi a la carpeta content.

# DATASET TIENDA MINORISTA ONLINE

# Fase I: Formulación del Problema y Entendimiento de los Datos

## Definición del Problema de Negocio

### Contexto
Transacciones de una tienda minorista online del Reino Unido entre 01/12/2009 y 09/12/2011. Contiene información sobre facturas, productos, cantidades, precios, clientes y países.

### Pregunta de investigación
¿Es posible predecir si un cliente realizará compras de alto valor,
basándonos en su comportamiento histórico de compras?

### Variable objetivo
Cliente_Alto_Valor — Variable binaria:
- 1 = Cliente cuyo gasto total supera la mediana de todos los clientes
- 0 = Cliente por debajo de esa mediana

### Relevancia para el negocio
Permite priorizar recursos (descuentos, campañas, atención
personalizada) hacia clientes con mayor potencial de retorno.

### Tipo de problema
Clasificación binaria supervisada

### Modelos a utilizar (en orden de complejidad)
1. Regresión Logística  → modelo base, interpretable
2. Árbol de Decisión    → visual, fácil de explicar
3. Random Forest        → mayor precisión

### Métrica de evaluación principal
F1-Score — porque nos importa tanto no perder clientes valiosos
(recall) como no clasificar mal a los de bajo valor (precision)

### Unidad de análisis
Un cliente (Customer ID) con su historial completo de compras

**NOTA IMPORTANTE** — Limitación del dataset:

El dataset original (UCI) cubre 2009-2011 con 1,067,371
transacciones distribuidas en dos hojas Excel:
- Hoja 1: Year 2009-2010
- Hoja 2: Year 2010-2011

Por limitaciones de acceso al archivo completo, este análisis
trabaja únicamente con la Hoja 1 (Year 2009-2010), que contiene
405,752 transacciones limpias correspondientes a un año completo
de operaciones (01/12/2009 - 09/12/2010).

## 1.1 Carga de librerías

In [ ]:
# Librerías básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configuración
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Para que pandas muestre más columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1.2 Carga de datos

In [ ]:
# Cargar datos
retail_df = pd.read_excel('/content/online_retail_II.xlsx', na_values='na')
# Información general
print(f"Shape: {retail_df.shape}")
print(f"\nTipos de datos:")
print(retail_df.dtypes)
print(f"\nPrimeras filas:")
retail_df.head()

**NOTA**: Hay que evaluar cambio de Customer_ID a variable categórica

IDENTIFICAMOS LOS VALORES NULOS Y FALTANTES

In [ ]:
# Identificar la cantidad de valores nulos (NaN) por columna en el DataFrame
missing_values_count = retail_df.isnull().sum()

# Calcular el porcentaje de valores nulos por columna
missing_values_percentage = (retail_df.isnull().sum() / len(retail_df)) * 100

# Unir ambas Series en un DataFrame para una mejor visualización
missing_values_table = pd.DataFrame({
    'Count': missing_values_count,
    'Percentage': missing_values_percentage
})

# Filtrar solo las columnas que tienen valores nulos y ordenarlas
missing_values_table = missing_values_table[missing_values_table['Count'] > 0].sort_values(by='Count', ascending=False)

print("Valores faltantes por columna en el DataFrame:")
display(missing_values_table)

**NOTA**: Description podemos eliminar porque no es informacion relevante, los consumidores que no tienen ID también vamos a eliminarlo para poder construir perfiles de clientes.

In [ ]:
# Filtrar el DataFrame para obtener solo las filas con valores nulos en 'Description'
desc_missing_df = retail_df[retail_df['Description'].isnull()]

# Contar la frecuencia de cada StockCode en estas filas
stockcode_missing_counts = desc_missing_df['StockCode'].value_counts()

print("Top 10 StockCodes con la mayor cantidad de descripciones faltantes:")
display(stockcode_missing_counts.head(10))

Los valores faltantes no estan asociados a un producto en particular, puede ser falla al ingresar los datos, o falla del sistema

**DESICIÓN**: Eliminar las filas de los datos faltantes ya que no representan un patrón y nos pueden causar problemas futuros.

In [ ]:
print(f"Filas antes: {len(retail_df)}")

retail_df = retail_df.dropna(subset=['Description'])

print(f"Filas después: {len(retail_df)}")

In [ ]:
retail_df = retail_df.dropna(subset=['Customer ID'])

print(f"Filas después de eliminar Customer ID nulo: {len(retail_df)}")

COMPRUEBO QUE EL DATASET ESTA LIMPIO

In [ ]:
# Identificar la cantidad de valores nulos (NaN) por columna en el DataFrame
missing_values_count = retail_df.isnull().sum()

# Calcular el porcentaje de valores nulos por columna
missing_values_percentage = (retail_df.isnull().sum() / len(retail_df)) * 100

# Unir ambas Series en un DataFrame para una mejor visualización
missing_values_table = pd.DataFrame({
    'Count': missing_values_count,
    'Percentage': missing_values_percentage
})

# Filtrar solo las columnas que tienen valores nulos y ordenarlas
missing_values_table = missing_values_table[missing_values_table['Count'] > 0].sort_values(by='Count', ascending=False)

print("Valores faltantes por columna en el DataFrame:")
display(missing_values_table)

## 1.3 Unidad de observación

ANALIZAR DUPLICADOS

In [ ]:
# Verificar duplicados
print(f"Total de filas: {len(retail_df)}")
print(f"Usuarios únicos: {retail_df['Customer ID'].nunique()}")

# Contar usuarios con más de un registro
duplicados = retail_df['Customer ID'].value_counts()
usuarios_duplicados = duplicados[duplicados > 1]
print(f"\nUsuarios con más de un registro: {len(usuarios_duplicados)}")

In [ ]:
# Contar el número de países únicos
num_paises_unicos = retail_df['Country'].nunique()
print(f"Número de países únicos en el DataFrame: {num_paises_unicos}")

In [ ]:
# Obtener la frecuencia de países en orden decreciente
conteo_decreciente = retail_df['Country'].value_counts()

# Mostrar la lista de países y su frecuencia de compra en orden decreciente
display(conteo_decreciente.sort_values(ascending=False))

In [ ]:
usuarios_duplicados.head()

In [ ]:
# Ejemplos de usuarios duplicados
print(f"\nEjemplo - Usuario 14156.0	:")
(retail_df[retail_df['Customer ID'] == 14156.0	])

In [ ]:
# Obtener los 5 Customer IDs con más transacciones
top_5_customer_ids = retail_df['Customer ID'].value_counts().head(5).index

print("Mostrando las primeras 5 compras para los 5 clientes con más transacciones:\n")

# Iterar sobre los IDs de cliente y mostrar sus primeras 5 compras
for customer_id in top_5_customer_ids:
    print(f"--- Compras del Customer ID: {customer_id} ---")
    customer_purchases = retail_df[retail_df['Customer ID'] == customer_id]
    display(customer_purchases.head(10))
    print("\n")

**NOTA**: Los valores duplicados significa que, **son usuarios con compras recurrentes**, interesante para análsis futuros

In [ ]:
# Asegurar que la columna 'TotalPrice' exista antes de usarla
retail_df['TotalPrice'] = retail_df['Quantity'] * retail_df['Price']

# Calcular el valor monetario total por cliente
clientes_monetario = retail_df.groupby('Customer ID')['TotalPrice'].sum()

# Obtener los Customer IDs del top 10 de clientes por valor monetario
top_10_monetary_customer_ids = clientes_monetario.nlargest(10).index.tolist()

print("Países de los Top 10 Clientes con mayores volúmenes de compra (basado en valor monetario):\n")

# Iterar sobre cada Customer ID en el top 10
for customer_id in top_10_monetary_customer_ids:
    # Filtrar las transacciones de este cliente
    customer_transactions = retail_df[retail_df['Customer ID'] == customer_id]
    # Obtener los países únicos de las transacciones de este cliente
    countries = customer_transactions['Country'].unique()
    print(f"Customer ID {customer_id}: {', '.join(countries)}")

In [ ]:
# Filtrar las transacciones donde el país es 'Unspecified'
unspecified_country_transactions = retail_df[retail_df['Country'] == 'Unspecified']

print(f"Número de transacciones con País 'Unspecified': {len(unspecified_country_transactions)}")
print("\nValores únicos de 'Customer ID' en transacciones 'Unspecified':")
display(unspecified_country_transactions['Customer ID'].unique())

print("\nPrimeras 5 filas de transacciones con País 'Unspecified':")
display(unspecified_country_transactions.head())

**NOTA**: El número de transacciones de países no especificados SON minimos, por tanto se podria eliminar si los datos no son relevantes para el modelo

In [ ]:
# Contar la frecuencia de cada Customer ID en las transacciones 'Unspecified'
customers_in_unspecified_country = unspecified_country_transactions['Customer ID'].value_counts()

# Filtrar solo a los clientes con más de una compra en 'Unspecified' y mostrarlos junto con sus frecuencias
duplicate_customers_unspecified = customers_in_unspecified_country[customers_in_unspecified_country > 1]

print("Clientes con múltiples compras en transacciones de País 'Unspecified' y sus frecuencias:")
display(duplicate_customers_unspecified)

**DESICIÓN**: Eliminar Unspecific

In [ ]:
# Eliminamos Unspecific
retail_df = retail_df[retail_df['Country'] != 'Unspecified']

print(f"Filas después: {len(retail_df)}")

## 1.4 Coherencia de los datos

### 1.4.1 Variables categóricas

In [ ]:
# Identificar y listar las variables categóricas (tipo 'object')
var_categoricas = retail_df.select_dtypes(include=['object']).columns.tolist()
print(f"Las variables categóricas en el DataFrame son: {var_categoricas}")

In [ ]:
import matplotlib.pyplot as plt

# Obtener la frecuencia de países en orden decreciente
conteo_decreciente = retail_df['Country'].value_counts().sort_values(ascending=False)

# Definir el tamaño del grupo para la visualización
tamanio_grupo = 10
num_paises = len(conteo_decreciente)

print(f"Total de países: {num_paises}")
print(f"Se mostrarán en grupos de {tamanio_grupo} países.\n")

# Iterar y crear gráficos por grupos de 10 países
for i in range(0, num_paises, tamanio_grupo):
    inicio = i
    fin = min(i + tamanio_grupo, num_paises)
    grupo_paises = conteo_decreciente.iloc[inicio:fin]

    plt.figure(figsize=(10, 6))
    # Para que el país con mayor número de compras quede arriba en el gráfico barh,
    # el subgrupo debe ordenarse de forma ascendente antes de plotear.
    grupo_paises.sort_values(ascending=True).plot(kind='barh')
    plt.title(f'Frecuencia de Transacciones por País (Grupo {int(i/tamanio_grupo)+1}: Países {inicio+1}-{fin})')
    plt.xlabel('Frecuencia de Compras')
    plt.ylabel('País')
    plt.tight_layout()
    plt.show()
    print("\n")

### 1.4.2 Variables numéricas

In [ ]:
# Identificar y listar las variables numéricas (tipo 'int64' o 'float64')
var_numericas = retail_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Las variables numéricas en el DataFrame son: {var_numericas}")

In [ ]:
# Grupo 1: Variables 1-3
vars_grupo1 = var_numericas[0:3] # Tomamos las 3 variables numéricas
print(f"Grupo 1: {vars_grupo1}")

fig, axes = plt.subplots(1, len(vars_grupo1), figsize=(15, 4)) # Ajustamos el número de subplots
for i, var in enumerate(vars_grupo1):
    retail_df[var].hist(bins=30, ax=axes[i], edgecolor='black') # Usamos retail_df
    axes[i].set_title(var)
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

# Estadísticas descriptivas
print("\nEstadísticas descriptivas:")
display(retail_df[vars_grupo1].describe()) # Usamos retail_df

Desición: Cambiamos la variable Customer_ID de numérica a Categórica

## 1.4.3 Corrección de Tipos

Conversión de `Customer ID` a categórica

In [ ]:
# Convertir 'Customer ID' a tipo 'object' ya que es un identificador y no una variable numérica para cálculos.
retail_df['Customer ID'] = retail_df['Customer ID'].astype(str)

# Verificar el tipo de datos después de la conversión
print("Tipo de dato de 'Customer ID' después de la conversión:")
print(retail_df['Customer ID'].dtype)

# Actualizar la lista de variables numéricas y categóricas para reflejar el cambio
var_numericas = retail_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
var_categoricas = retail_df.select_dtypes(include=['object']).columns.tolist()

print(f"\nNuevas variables numéricas: {var_numericas}")
print(f"Nuevas variables categóricas: {var_categoricas}")

Le cambiamos la variable categórica a variable str

In [ ]:
# Customer ID ya es categórica, la convertimos a str para mayor compatibilidad
retail_df['Customer ID'] = retail_df['Customer ID'].astype(str)
retail_df['InvoiceDate'] = pd.to_datetime(retail_df['InvoiceDate'])

print(retail_df.dtypes)

In [ ]:
# Identificar y listar las variables numéricas (tipo 'int64' o 'float64')
var_numericas = retail_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Las variables numéricas en el DataFrame son: {var_numericas}")

In [ ]:
# Grupo 1: Variables 1-2
vars_grupo1 = var_numericas[0:3] # Ahora si Tomamos las 2 variables numéricas
print(f"Grupo 1: {vars_grupo1}")

fig, axes = plt.subplots(1, len(vars_grupo1), figsize=(15, 4)) # Ajustamos el número de subplots
for i, var in enumerate(vars_grupo1):
    retail_df[var].hist(bins=30, ax=axes[i], edgecolor='black') # Usamos retail_df
    axes[i].set_title(var)
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

# Estadísticas descriptivas
print("\nEstadísticas descriptivas:")
display(retail_df[vars_grupo1].describe()) # Usamos retail_df

NOTA: Luego tomaremos una desición con los datos de cantidad y precio, ya que no deberían ser negativos

ANÁLISIS DE VALORES NEGATIVOS

In [ ]:
## TRANSACCIONES NEGATIVAS

import pandas as pd

# Filtrar transacciones con Quantity negativa
negative_quantity_transactions = retail_df[retail_df['Quantity'] < 0]

# Obtener los Customer IDs únicos de estas transacciones
customers_with_negative_quantity = negative_quantity_transactions['Customer ID'].astype(str).unique()

print(f"Número de transacciones con Quantity negativa: {len(negative_quantity_transactions)}")
print("\nCustomer IDs con transacciones de Quantity negativa:")
display(customers_with_negative_quantity)

print("\nPaíses con transacciones de Quantity negativa:")
display(negative_quantity_transactions['Country'].unique())

print("\nPrimeras 5 filas de transacciones con Quantity negativa (para ejemplo):")
display(negative_quantity_transactions.head())

In [ ]:
# Contar las transacciones negativas por país y obtener el top 10
top_10_paises_negativos = negative_quantity_transactions['Country'].value_counts().head(10)

print("Top 10 Países con más transacciones de Quantity negativa:")
display(top_10_paises_negativos)

In [ ]:
# Contar las transacciones negativas por producto (Description) y obtener el top 10
top_10_productos_negativos = negative_quantity_transactions['Description'].value_counts().head(10)

print("Top 10 Productos con más transacciones de Quantity negativa:")
display(top_10_productos_negativos)

In [ ]:
# Identificar todos los países en el dataset
all_countries = retail_df['Country'].unique()

# Identificar los países que tienen al menos una transacción con Quantity negativa
countries_with_negative_transactions = negative_quantity_transactions['Country'].unique()

# Encontrar los países que NO tienen transacciones con Quantity negativa
countries_without_negative_transactions = [country for country in all_countries if country not in countries_with_negative_transactions]

# Filtrar el DataFrame original para incluir solo las transacciones de estos países
retail_df_without_negatives = retail_df[retail_df['Country'].isin(countries_without_negative_transactions)]

# Contar la frecuencia de transacciones para estos países y obtener el top 10
top_10_paises_sin_negativos = retail_df_without_negatives['Country'].value_counts().head(10)

print("Top 10 Países SIN transacciones de Quantity negativa (y su frecuencia de compra total):")
display(top_10_paises_sin_negativos)

In [ ]:
# Identificar todas las descripciones de productos en el dataset completo
all_product_descriptions = retail_df['Description'].unique()

# Identificar las descripciones de productos que tienen al menos una transacción con Quantity negativa
products_with_negative_transactions = negative_quantity_transactions['Description'].unique()

# Encontrar las descripciones de productos que NO tienen transacciones con Quantity negativa
products_without_negative_transactions = [desc for desc in all_product_descriptions if desc not in products_with_negative_transactions]

# Filtrar el DataFrame original para incluir solo las transacciones de estos productos
retail_df_without_product_negatives = retail_df[retail_df['Description'].isin(products_without_negative_transactions)]

# Contar la frecuencia de transacciones para estos productos y obtener el top 10
top_10_productos_sin_negativos = retail_df_without_product_negatives['Description'].value_counts().head(10)

print("Top 10 Productos SIN transacciones de Quantity negativa (y su frecuencia de compra total):")
display(top_10_productos_sin_negativos)

In [ ]:
# Contar las transacciones negativas por Customer ID y obtener el top 10
top_10_clientes_negativos = negative_quantity_transactions['Customer ID'].value_counts().head(10)

print("Top 10 Clientes con más transacciones de Quantity negativa:")
display(top_10_clientes_negativos)

In [ ]:
import re

# Filtrar invoices que contengan letras (no son puramente numéricos)
alphanumeric_invoices_df = retail_df[retail_df['Invoice'].astype(str).str.contains('[a-zA-Z]', regex=True)].copy()

# Filtrar adicionalmente para asegurar que también contengan números
alphanumeric_invoices_df = alphanumeric_invoices_df[alphanumeric_invoices_df['Invoice'].astype(str).str.contains('[0-9]', regex=True)]

print(f"Número de invoices alfanuméricos: {len(alphanumeric_invoices_df)}")

# Extraer todas las letras de los invoices alfanuméricos
all_letters = ''.join(alphanumeric_invoices_df['Invoice'].astype(str).str.findall('[a-zA-Z]').sum())

# Contar la frecuencia de cada letra
from collections import Counter
letter_counts = Counter(all_letters)

print("\nFrecuencia de las letras más usuales en invoices alfanuméricos:")
for letter, count in letter_counts.most_common(10):
    print(f"'{letter}': {count}")

**NOTA**: El análisis de la columna `Invoice` ha revelado que muchas de las transacciones con cantidades negativas tienen una 'C' al inicio y algunas una letra A. Las letras A junto al número de la factura pueden ser errores tipográficos. Las letras C junto al número de la factura deben ser las transacciones canceladas.

**DESICIÓN**: Eliminar transacciones Canceladas, es decir los valores numéricos negativos. INVOICES con letra C y A

In [ ]:
import numpy as np

# Filtrar las transacciones con Invoice que empiezan por 'C'
c_invoices_transactions = retail_df[retail_df['Invoice'].astype(str).str.startswith('C')]

print(f"Número de transacciones con Invoice que empiezan con 'C': {len(c_invoices_transactions)}")
print(f"Cantidad Mínima para Invoices 'C': {c_invoices_transactions['Quantity'].min()}")
print(f"Cantidad Máxima para Invoices 'C': {c_invoices_transactions['Quantity'].max()}")
print(f"Cantidad Promedio para Invoices 'C': {c_invoices_transactions['Quantity'].mean():.2f}")

# Filtrar las transacciones con Invoice que empiezan por 'A' (las raras)
a_invoices_transactions = retail_df[retail_df['Invoice'].astype(str).str.startswith('A')]

print(f"\nNúmero de transacciones con Invoice que empiezan con 'A': {len(a_invoices_transactions)}")
if not a_invoices_transactions.empty:
    print(f"Cantidad Mínima para Invoices 'A': {a_invoices_transactions['Quantity'].min()}")
    print(f"Cantidad Máxima para Invoices 'A': {a_invoices_transactions['Quantity'].max()}")
    print(f"Cantidad Promedio para Invoices 'A': {a_invoices_transactions['Quantity'].mean():.2f}")
else:
    print("No hay transacciones con Invoice que empiezan con 'A' para analizar la cantidad.")

print("\nPorcentaje de transacciones con 'C' que tienen Quantity negativa:")
negative_c_invoices_count = c_invoices_transactions[c_invoices_transactions['Quantity'] < 0].shape[0]
total_c_invoices_count = c_invoices_transactions.shape[0]
if total_c_invoices_count > 0:
    percentage = (negative_c_invoices_count / total_c_invoices_count) * 100
    print(f"{percentage:.2f}% de las transacciones con 'C' tienen Quantity negativa.")
else:
    print("No hay transacciones con Invoice que empiezan con 'C'.")

**CONFIRMADO**: Las **transacciones negativas** corresponden a Cancelaciones

In [ ]:
import re
from collections import Counter

# Filtrar StockCode que contengan letras
alphanumeric_stockcodes_df = retail_df[retail_df['StockCode'].astype(str).str.contains('[a-zA-Z]', regex=True)].copy()

print(f"Número de StockCodes alfanuméricos: {len(alphanumeric_stockcodes_df)}")

# Extraer todas las letras de los StockCodes alfanuméricos
all_stockcode_letters = ''.join(alphanumeric_stockcodes_df['StockCode'].astype(str).str.findall('[a-zA-Z]').sum())

# Contar la frecuencia de cada letra
stockcode_letter_counts = Counter(all_stockcode_letters)

print("\nFrecuencia de las letras más usuales en StockCodes alfanuméricos:")
for letter, count in stockcode_letter_counts.most_common(10):
    print(f"'{letter}': {count}")

Las letras junto a **STOCKCODES** pueden ser parte de la estructura del codigo de producto

**DESICIÓN**: Eliminar transacciones Canceladas, es decir los valores numéricos negativos. INVOICES con letra C y A

In [ ]:
# Facturas que empiezan con 'C' = cancelaciones
# Facturas que empiezan con 'A' = posibles errores tipográficos
# Ambas están asociadas a cantidades negativas

mask_canceladas = retail_df['Invoice'].astype(str).str.startswith('C')
mask_errores    = retail_df['Invoice'].astype(str).str.startswith('A')

print(f"Transacciones con 'C' (cancelaciones): {mask_canceladas.sum()}")
print(f"Transacciones con 'A' (errores):        {mask_errores.sum()}")
print(f"Total a eliminar:                        {(mask_canceladas | mask_errores).sum()}")

retail_df = retail_df[~(mask_canceladas | mask_errores)]

print(f"\nFilas después de eliminar C y A: {len(retail_df)}")

VERIFICAMOS QUE PRECIO Y CANTIDAD NO TENGAN ERRORES

In [ ]:
retail_df = retail_df[retail_df['Price'] > 0]
retail_df = retail_df[retail_df['Quantity'] > 0]

print(f"Filas después: {len(retail_df)}")

AGREGAMOS UNA COLUMNA IMPORTANTE

In [ ]:
retail_df['TotalPrice'] = retail_df['Quantity'] * retail_df['Price']

print(f"TotalPrice - min:   {retail_df['TotalPrice'].min():.2f}")
print(f"TotalPrice - max:   {retail_df['TotalPrice'].max():.2f}")
print(f"TotalPrice - media: {retail_df['TotalPrice'].mean():.2f}")

## 1.5 Guardado de datos preparados

In [ ]:
# Resumen final
print("=== RESUMEN DE LIMPIEZA ===")
print(f"Filas finales:    {len(retail_df)}")
print(f"Clientes únicos:  {retail_df['Customer ID'].nunique()}")
print(f"Países únicos:    {retail_df['Country'].nunique()}")
print(f"\nValores nulos restantes:")
print(retail_df.isnull().sum())

import os
# Definir el nombre de la carpeta y el archivo
output_dir = 'prepared'
output_filename = 'online_retail_II_prepared.xlsx'
output_path = os.path.join(output_dir, output_filename)

# Crear la carpeta 'prepared' si no existe
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Carpeta '{output_dir}' creada.")

# Guardar
retail_df.to_excel(output_path, index=False)
print(f"Datos preparados guardados exitosamente en: {output_path}")

In [ ]:
# Verificación final del estado del dataset
print("=== ESTADO FINAL DEL DATASET ===")
print(f"Shape:            {retail_df.shape}")
print(f"Nulos restantes:\n{retail_df.isnull().sum()}")
print(f"Quantity mínima:  {retail_df['Quantity'].min()}")
print(f"Price mínimo:     {retail_df['Price'].min()}")
print(f"Facturas con C:   {retail_df['Invoice'].astype(str).str.startswith('C').sum()}")
print(f"Facturas con A:   {retail_df['Invoice'].astype(str).str.startswith('A').sum()}")
print(f"'TotalPrice' existe: {'TotalPrice' in retail_df.columns}")
print(f"Tipo Customer ID: {retail_df['Customer ID'].dtype}")
print(f"Tipo InvoiceDate: {retail_df['InvoiceDate'].dtype}")

# 2. Fase II: Preparación y Exploración de Datos (EDA)

2.1 Análisis de Outliers

In [ ]:
# Visualización con Boxplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].boxplot(retail_df['Quantity'], flierprops=dict(marker='o', markersize=2))
axes[0].set_title('Boxplot - Quantity')
axes[0].set_ylabel('Valor')

axes[1].boxplot(retail_df['Price'], flierprops=dict(marker='o', markersize=2))
axes[1].set_title('Boxplot - Price')
axes[1].set_ylabel('Valor')

axes[2].boxplot(retail_df['TotalPrice'], flierprops=dict(marker='o', markersize=2))
axes[2].set_title('Boxplot - TotalPrice')
axes[2].set_ylabel('Valor')

plt.tight_layout()
plt.show()

In [ ]:
# Cuantificar outliers con IQR
def analizar_outliers(df, columna):
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    outliers = df[(df[columna] < limite_inferior) | (df[columna] > limite_superior)]

    print(f"=== {columna} ===")
    print(f"Q1:                {Q1:.2f}")
    print(f"Q3:                {Q3:.2f}")
    print(f"IQR:               {IQR:.2f}")
    print(f"Límite superior:   {limite_superior:.2f}")
    print(f"Outliers:          {len(outliers)} filas ({len(outliers)/len(df)*100:.1f}%)")
    print(f"Valor máximo:      {df[columna].max():.2f}")
    print()

analizar_outliers(retail_df, 'Quantity')
analizar_outliers(retail_df, 'Price')
analizar_outliers(retail_df, 'TotalPrice')

Las transacciones con Quantity > 27 unidades
podrían corresponder a clientes mayoristas o empresas.
Se mantienen en el dataset ya que representan
comportamiento de compra real, no errores.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

variables = ['Quantity', 'Price', 'TotalPrice']

for i, var in enumerate(variables):
    # Fila 1: Distribución completa
    axes[0, i].hist(retail_df[var], bins=50, edgecolor='black', color='steelblue')
    axes[0, i].set_title(f'{var} — Distribución Completa')
    axes[0, i].set_xlabel('Valor')
    axes[0, i].set_ylabel('Frecuencia')

    # Fila 2: Distribución sin outliers extremos (percentil 99)
    p99 = retail_df[var].quantile(0.99)
    datos_filtrados = retail_df[retail_df[var] <= p99][var]
    axes[1, i].hist(datos_filtrados, bins=50, edgecolor='black', color='coral')
    axes[1, i].set_title(f'{var} — Sin top 1% (hasta £{p99:.0f})')
    axes[1, i].set_xlabel('Valor')
    axes[1, i].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

In [ ]:
# Estadísticas descriptivas completas
print("=== ESTADÍSTICAS DESCRIPTIVAS ===\n")
for var in ['Quantity', 'Price', 'TotalPrice']:
    print(f"--- {var} ---")
    print(f"Media:      {retail_df[var].mean():.2f}")
    print(f"Mediana:    {retail_df[var].median():.2f}")
    print(f"Percentil 90: {retail_df[var].quantile(0.90):.2f}")
    print(f"Percentil 95: {retail_df[var].quantile(0.95):.2f}")
    print(f"Percentil 99: {retail_df[var].quantile(0.99):.2f}")
    print(f"Máximo:     {retail_df[var].max():.2f}")
    print()

In [ ]:
# Ver quiénes son los clientes con compras masivas
compras_masivas = retail_df[retail_df['Quantity'] > 27]

print(f"Total compras masivas: {len(compras_masivas)}")
print(f"\nTop 10 clientes con más compras masivas:")
display(compras_masivas.groupby('Customer ID')['Quantity'].sum()
        .sort_values(ascending=False).head(10))

print(f"\nProductos más comprados en grandes cantidades:")
display(compras_masivas.groupby('Description')['Quantity'].sum()
        .sort_values(ascending=False).head(10))

print(f"\nPaíses con más compras masivas:")
display(compras_masivas['Country'].value_counts().head(10))

ANALIZAMOS PRODUCTOS SOSPECHOSOS

In [ ]:
# ¿Qué producto cuesta £10,953?
productos_caros = retail_df[retail_df['Price'] > 1000]

print(f"Transacciones con Price > £1,000: {len(productos_caros)}")
print("\nProductos:")
display(productos_caros[['StockCode', 'Description', 'Price', 'Quantity', 'Country']].drop_duplicates())

# ## Decisión preliminar
# Quantity  # MANTENER  (mayoristas reales)
# TotalPrice # MANTENER (consecuencia de lo anterior)
# Price      # INVESTIGAR primero, luego decidir

STOCKCODE TERMINA EN M: Deben operaciones manuales que se realizan en caja para corregir errores o hacer ajustes. No son producrtos reales. Siempre se hacen por unidad y van desde 1000usd hasta 10953usd, tienen un mismo patrón.

**DESICIÓN**: Eliminar transacciones con StockCode que termina en 'M' (ajustes manuales)

In [ ]:
# Eliminar transacciones con StockCode que termina en 'M' (ajustes manuales)
manual_mask = retail_df['StockCode'].astype(str).str.endswith('M')

print(f"Transacciones manuales a eliminar: {manual_mask.sum()}")

retail_df = retail_df[~manual_mask]

print(f"Filas después de eliminar manuales: {len(retail_df)}")

In [ ]:
print(f"Price máximo ahora: £{retail_df['Price'].max():.2f}")
print(f"Top 5 precios más altos:")
display(retail_df.nlargest(5, 'Price')[['StockCode','Description','Price','Quantity']])

APARECEN NUEVOS PRECIOS: Estos precios son los generados hacia el cliente por gastos de envío, mas no son por venta de producto. Son correcciones manuales hecjhas por empleados, que no representan ventas reales.

**DESICIÓN**: Eliminar esos valores que corresponden a POSTAGE y ADJUSTMENTS

In [ ]:
# Eliminar POSTAGE y ADJUSTMENTS
codigos_especiales = ['POST', 'ADJUST', 'ADJUST2']

mask_especiales = retail_df['StockCode'].astype(str).isin(codigos_especiales)

print(f"Transacciones especiales a eliminar: {mask_especiales.sum()}")

retail_df = retail_df[~mask_especiales]

print(f"Filas después: {len(retail_df)}")

# Verificar precio máximo ahora
print(f"\nPrice máximo ahora: £{retail_df['Price'].max():.2f}")
print(f"\nTop 5 precios más altos:")
display(retail_df.nlargest(5, 'Price')[['StockCode','Description','Price','Quantity']])

In [ ]:
# VERIFICACIÓN DE LIMPIEZA
# Verificación completa del estado actual
analizar_outliers(retail_df, 'Quantity')
analizar_outliers(retail_df, 'Price')
analizar_outliers(retail_df, 'TotalPrice')

print(f"\nEstadísticas finales:")
print(f"Price máximo:     £{retail_df['Price'].max():.2f}")
print(f"Quantity máxima:  {retail_df['Quantity'].max()}")
print(f"TotalPrice máx:   £{retail_df['TotalPrice'].max():.2f}")

**NOTA OUTLIERS:**
- Price: resuelto, máximo £295 corresponde a producto real
- Quantity: se mantienen valores extremos (>27 unidades)
  ya que podrían corresponder a clientes mayoristas
- TotalPrice: se mantiene como consecuencia de Quantity
- El modelo deberá ser robusto ante estos valores extremos,
  para lo cual aplicaremos transformación logarítmica
  en el Feature Engineering

**FEATURES ENGINEERING**

In [ ]:
# La fecha de referencia es el último día del dataset
# A partir de aquí calculamos la Recencia
fecha_referencia = retail_df['InvoiceDate'].max()
print(f"Fecha de referencia: {fecha_referencia}")

Construir la tabla RFM

In [ ]:
rfm = retail_df.groupby('Customer ID').agg(
    Recency    = ('InvoiceDate', lambda x: (fecha_referencia - x.max()).days),
    Frequency  = ('Invoice',     'nunique'),
    Monetary   = ('TotalPrice',  'sum')
).reset_index()

print(f"Shape de la tabla RFM: {rfm.shape}")
print(f"\nPrimeras filas:")
display(rfm.head(10))
print(f"\nEstadísticas descriptivas:")
display(rfm.describe())

In [ ]:
print("=== VERIFICACIÓN RFM ===")
print(f"Clientes únicos:      {rfm['Customer ID'].nunique()}")
print(f"Recency mínima:       {rfm['Recency'].min()} días")
print(f"Recency máxima:       {rfm['Recency'].max()} días")
print(f"Frequency mínima:     {rfm['Frequency'].min()} compras")
print(f"Frequency máxima:     {rfm['Frequency'].max()} compras")
print(f"Monetary mínima:      £{rfm['Monetary'].min():.2f}")
print(f"Monetary máxima:      £{rfm['Monetary'].max():.2f}")
print(f"Monetary mediana:     £{rfm['Monetary'].median():.2f}")

Cliente Alto Valor  → gasta más de £702.90 al año → 1
Cliente Bajo Valor  → gasta menos de £702.90 al año → 0

CREAR VARIABLE OBJETIVO

In [ ]:
# Crear variable objetivo basada en la mediana
mediana_monetary = rfm['Monetary'].median()
print(f"Mediana de Monetary: £{mediana_monetary:.2f}")

rfm['Cliente_Alto_Valor'] = (rfm['Monetary'] > mediana_monetary).astype(int)

# Verificar balance de clases
print(f"\nDistribución de la variable objetivo:")
print(rfm['Cliente_Alto_Valor'].value_counts())
print(f"\nPorcentaje:")
print(rfm['Cliente_Alto_Valor'].value_counts(normalize=True) * 100)

Tenemos un Dataset Balanceado

AGREGAMOS 2 FEATURES ADICIONALES QUE SON NECESARIOS

In [ ]:
# Feature 1: Ticket promedio por compra
rfm['Ticket_Promedio'] = rfm['Monetary'] / rfm['Frequency']

# Feature 2: Variedad de productos distintos comprados
productos_distintos = retail_df.groupby('Customer ID')['StockCode'].nunique().reset_index()
productos_distintos.columns = ['Customer ID', 'Productos_Distintos']

# Feature 3: País del cliente
pais_cliente = retail_df.groupby('Customer ID')['Country'].first().reset_index()

# Unir todo a la tabla RFM
rfm = rfm.merge(productos_distintos, on='Customer ID')
rfm = rfm.merge(pais_cliente, on='Customer ID')

print(f"Shape final RFM: {rfm.shape}")
print(f"\nColumnas: {rfm.columns.tolist()}")
display(rfm.head())

GUARDAMOS LA TABLA RFM

In [ ]:
import os

output_dir = 'prepared'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

rfm.to_excel('prepared/rfm_clientes.xlsx', index=False)
print("Tabla RFM guardada exitosamente")
print(f"\nResumen final:")
print(f"Clientes:             {len(rfm)}")
print(f"Features:             {rfm.shape[1] - 2}")
print(f"Variable objetivo:    Cliente_Alto_Valor")
print(f"Balance clases:       50% / 50%")

2.2 Analisis Univariado

In [ ]:
# VARIABLES NUMÉRICAS
import matplotlib.pyplot as plt
import seaborn as sns

variables_numericas = ['Recency', 'Frequency', 'Monetary',
                       'Ticket_Promedio', 'Productos_Distintos']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for i, var in enumerate(variables_numericas):
    # Fila 1: Histogramas
    axes[0, i].hist(rfm[var], bins=30, edgecolor='black', color='steelblue')
    axes[0, i].set_title(f'{var}\nHistograma')
    axes[0, i].set_xlabel('Valor')
    axes[0, i].set_ylabel('Frecuencia')

    # Fila 2: Boxplots
    axes[1, i].boxplot(rfm[var], flierprops=dict(marker='o', markersize=2))
    axes[1, i].set_title(f'{var}\nBoxplot')
    axes[1, i].set_ylabel('Valor')

plt.tight_layout()
plt.show()

In [ ]:
# Estadísticas descriptivas completas
print("=== ESTADÍSTICAS DESCRIPTIVAS — TABLA RFM ===\n")
for var in variables_numericas:
    print(f"--- {var} ---")
    print(f"Media:        {rfm[var].mean():.2f}")
    print(f"Mediana:      {rfm[var].median():.2f}")
    print(f"Desv. Std:    {rfm[var].std():.2f}")
    print(f"Mínimo:       {rfm[var].min():.2f}")
    print(f"Máximo:       {rfm[var].max():.2f}")
    print(f"Sesgo:        {rfm[var].skew():.2f}")
    print()

Debido a que los sesgos son muy altos, debemos aplicar transformación logarìtmica.

In [ ]:
# Aplicar transformación log a variables con sesgo > 2
# log1p = log(x+1) para evitar log(0) que es infinito

rfm['Recency_log']             = np.log1p(rfm['Recency'])
rfm['Frequency_log']           = np.log1p(rfm['Frequency'])
rfm['Monetary_log']            = np.log1p(rfm['Monetary'])
rfm['Ticket_Promedio_log']     = np.log1p(rfm['Ticket_Promedio'])
rfm['Productos_Distintos_log'] = np.log1p(rfm['Productos_Distintos'])

# Comparar sesgo antes y después
print("=== IMPACTO DE LA TRANSFORMACIÓN LOGARÍTMICA ===\n")
variables = ['Recency', 'Frequency', 'Monetary',
             'Ticket_Promedio', 'Productos_Distintos']

for var in variables:
    sesgo_antes  = rfm[var].skew()
    sesgo_despues = rfm[f'{var}_log'].skew()
    print(f"{var}")
    print(f"  Sesgo antes:   {sesgo_antes:.2f}")
    print(f"  Sesgo después: {sesgo_despues:.2f}")
    print()

Las variables logarítmicas se usarán para alimentar el modelo. Se ha logrado disminuir el sesgo hacia lo ideal.

In [ ]:
# Visualizar el antes y después
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for i, var in enumerate(variables):
    # Fila 1: Original
    axes[0, i].hist(rfm[var], bins=30,
                    edgecolor='black', color='steelblue')
    axes[0, i].set_title(f'{var}\nSesgo: {rfm[var].skew():.2f}')

    # Fila 2: Transformada
    axes[1, i].hist(rfm[f'{var}_log'], bins=30,
                    edgecolor='black', color='coral')
    axes[1, i].set_title(f'{var}_log\nSesgo: {rfm[f"{var}_log"].skew():.2f}')

plt.suptitle('Antes vs Después — Transformación Logarítmica',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# VARIABLE CATEGÓRICA — País
plt.figure(figsize=(12, 5))
top_paises = rfm['Country'].value_counts().head(10)
top_paises.sort_values().plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Top 10 Países por Número de Clientes')
plt.xlabel('Número de Clientes')
plt.ylabel('País')
plt.tight_layout()
plt.show()

print(f"\nDistribución de clientes por país (Top 10):")
print(f"Total países: {rfm['Country'].nunique()}")
display(top_paises)

Transformamos: En lugar de 36 países, creamos una variable binaria - Uk vs resto del mundo

In [ ]:
# Simplificar país a UK vs Resto del mundo
rfm['Es_UK'] = (rfm['Country'] == 'United Kingdom').astype(int)

print(rfm['Es_UK'].value_counts())
print(f"\nUK:              {rfm['Es_UK'].sum()} clientes")
print(f"Resto del mundo: {(rfm['Es_UK'] == 0).sum()} clientes")

# NOTA SOBRE CODIFICACIÓN DE VARIABLES CATEGÓRICAS
La única variable categórica relevante es 'Country'.
En lugar de aplicar One-Hot Encoding (que generaría 36 columnas
con muy pocos datos en la mayoría), se aplicó una codificación
binaria manual: Es_UK = 1 si el cliente es del Reino Unido, 0 si no.
Justificación: el 92.3% de los clientes son UK, por lo que
OHE generaría columnas altamente desbalanceadas que aportarían
ruido al modelo. La variable binaria captura la diferencia


**CONCLUSIONES UNIVARIADAS:**

1. Todas las variables numéricas presentan sesgo positivo
   importante, especialmente Monetary (24.23) y Frequency (9.93),
   lo que indica la presencia de clientes mayoristas con
   comportamiento muy diferente al cliente típico.

2. Se aplicó transformación logarítmica a todas las variables
   numéricas, reduciendo el sesgo a valores entre -0.61 y 1.18,
   rango aceptable para el modelado.

3. El 92.3% de los clientes son del Reino Unido, por lo que
   la variable Country se simplificó a una variable binaria
   Es_UK para evitar categorías con muy pocos datos.

4. La variable objetivo está perfectamente balanceada (50/50),
   lo que favorece el entrenamiento del modelo sin necesidad
   de técnicas de balanceo adicionales.

2.3 Análisis Bivariado

**Objetivo**: Ver si cada variable se comporta diferente
entre clientes Alto Valor (1) vs Bajo Valor (0)

In [ ]:
# Histogramas
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

variables_log = ['Recency_log', 'Frequency_log', 'Monetary_log',
                 'Ticket_Promedio_log', 'Productos_Distintos_log']

for i, var in enumerate(variables_log):
    row = i // 3
    col = i % 3

    grupo_0 = rfm[rfm['Cliente_Alto_Valor'] == 0][var]
    grupo_1 = rfm[rfm['Cliente_Alto_Valor'] == 1][var]

    axes[row, col].hist(grupo_0, bins=30, alpha=0.5,
                        color='steelblue', label='Bajo Valor (0)',
                        edgecolor='black')
    axes[row, col].hist(grupo_1, bins=30, alpha=0.5,
                        color='coral', label='Alto Valor (1)',
                        edgecolor='black')
    axes[row, col].set_title(f'{var} vs Cliente_Alto_Valor')
    axes[row, col].set_xlabel(var)
    axes[row, col].set_ylabel('Frecuencia')
    axes[row, col].legend()

axes[1, 2].axis('off')
plt.suptitle('Histogramas LOG — Variables vs Cliente Alto Valor',
             fontsize=14)
plt.tight_layout()
plt.show()

Los clientes de Alto Valor compraron más recientemente,
La separación es visible y clara;
→ Recency SUBE el riesgo de Bajo Valor
→ Predictor BUENO

Frecuency.- Los clientes de Bajo Valor compran UNA sola vez mayoritariamente
Los de Alto Valor tienen más compras recurrentes
Separación existe pero con mucho solapamiento
→ Predictor MODERADO

Monetary.- Separación PERFECTA entre los dos grupos
Casi sin solapamiento
→ Predictor MUY FUERTE
Monetary es la base de la variable objetivo
   Usar Monetary directamente en el modelo podría
   ser trampa (data leakage) — lo discutiremos **texto en negrita** texto en negrita **texto en negrita**

Ticket promedio_log.- Los clientes de Alto Valor gastan más por visita
Separación clara aunque con algo de solapamiento
→ Predictor BUENO

Productos Distintos.- Los clientes de Alto Valor compran más variedad de productos
Separación visible y consistente
→ Predictor BUENO

**DECISIÓN**: Excluir Monetary_log del modelo

**JUSTIFICACIÓN**:
La variable objetivo Cliente_Alto_Valor fue construida
directamente a partir de Monetary (gasto total > mediana).
Incluir Monetary como feature predictora constituye
data leakage, ya que el modelo tendría acceso directo
a la información con la que fue construida la variable
que intenta predecir.

Esto generaría métricas artificialmente altas que no
reflejarían la capacidad real del modelo para identificar
clientes de alto valor basándose en su comportamiento
de compra (Recency, Frequency, Ticket_Promedio,
Productos_Distintos).

Por tanto se excluye Monetary y Monetary_log del
conjunto de features para garantizar un modelo
honesto y generalizable.

In [ ]:
# Excluir Monetary y Monetary_log por data leakage
# La variable objetivo fue construida directamente desde Monetary

features_modelo = ['Recency_log',
                   'Frequency_log',
                   'Ticket_Promedio_log',
                   'Productos_Distintos_log',
                   'Es_UK']

# DataFrame final para el modelo
rfm_modelo = rfm[features_modelo + ['Cliente_Alto_Valor']].copy()

print("=== FEATURES FINALES DEL MODELO ===")
print(f"Shape: {rfm_modelo.shape}")
print(f"\nColumnas incluidas:")
for col in rfm_modelo.columns:
    if col == 'Cliente_Alto_Valor':
        print(f"  {col}  ← variable objetivo")
    else:
        print(f"  {col}  ← feature")

print(f"\nPrimeras filas:")
display(rfm_modelo.head())

print(f"\nValores nulos:")
print(rfm_modelo.isnull().sum())

In [ ]:
variables_log = ['Recency_log',
                 'Frequency_log',
                 'Ticket_Promedio_log',
                 'Productos_Distintos_log']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, var in enumerate(variables_log):
    row = i // 2
    col = i % 2

    grupo_0 = rfm[rfm['Cliente_Alto_Valor'] == 0][var]
    grupo_1 = rfm[rfm['Cliente_Alto_Valor'] == 1][var]

    axes[row, col].hist(grupo_0, bins=30, alpha=0.5,
                        color='steelblue', label='Bajo Valor (0)',
                        edgecolor='black')
    axes[row, col].hist(grupo_1, bins=30, alpha=0.5,
                        color='coral', label='Alto Valor (1)',
                        edgecolor='black')
    axes[row, col].set_title(f'{var} vs Cliente_Alto_Valor')
    axes[row, col].set_xlabel(var)
    axes[row, col].set_ylabel('Frecuencia')
    axes[row, col].legend()

plt.suptitle('Histogramas LOG — Variables vs Cliente Alto Valor',
             fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, var in enumerate(variables_log):
    row = i // 2
    col = i % 2

    grupo_0 = rfm[rfm['Cliente_Alto_Valor'] == 0][var]
    grupo_1 = rfm[rfm['Cliente_Alto_Valor'] == 1][var]

    bp = axes[row, col].boxplot([grupo_0, grupo_1],
                           labels=['Bajo Valor (0)', 'Alto Valor (1)'],
                           patch_artist=True)
    bp['boxes'][0].set_facecolor('steelblue')
    bp['boxes'][0].set_alpha(0.5)
    bp['boxes'][1].set_facecolor('coral')
    bp['boxes'][1].set_alpha(0.5)

    axes[row, col].set_title(f'{var} vs Cliente_Alto_Valor')
    axes[row, col].set_ylabel(var)

plt.suptitle('Boxplots LOG — Variables vs Cliente Alto Valor',
             fontsize=14)
plt.tight_layout()
plt.show()

Regency.- Bajo Valor  → caja entre 3.5 - 5.5  → compraron hace MUCHO
Alto Valor  → caja entre 2.0 - 4.5  → compraron más RECIENTEMENTE

Medianas:
Bajo Valor  → ~4.5
Alto Valor  → ~3.5

Frecuency.- Bajo Valor  → caja MUY pequeña entre 1.0 - 1.0
              → casi todos compraron solo 1 vez;
Alto Valor  → caja entre 1.0 - 2.5
              → compraron múltiples veces;

Medianas:
Bajo Valor  → ~1.0  (1 compra);
Alto Valor  → ~1.8  (casi el doble)

Ticket promedio.- Bajo Valor  → caja entre 4.5 - 6.0  → gasto bajo por visita;
Alto Valor  → caja entre 5.5 - 7.0  → gasto alto por visita;

Medianas:
Bajo Valor  → ~5.2;
Alto Valor  → ~6.0

Clientes distintos.- Bajo Valor  → caja entre 2.0 - 3.5  → poca variedad;
Alto Valor  → caja entre 3.5 - 5.0  → mucha variedad;

Medianas:
Bajo Valor  → ~3.0;
Alto Valor  → ~4.5

In [ ]:
print("=== ESTADÍSTICAS POR GRUPO (variables LOG) ===\n")

for var in variables_log:
    media_0 = rfm[rfm['Cliente_Alto_Valor'] == 0][var].mean()
    media_1 = rfm[rfm['Cliente_Alto_Valor'] == 1][var].mean()
    dif = ((media_1 - media_0) / media_0) * 100

    print(f"--- {var} ---")
    print(f"Bajo Valor  (0): {media_0:.2f}")
    print(f"Alto Valor  (1): {media_1:.2f}")
    print(f"Diferencia:      {dif:.1f}%")
    print()

In [ ]:
tabla_pais = pd.crosstab(rfm['Es_UK'],
                          rfm['Cliente_Alto_Valor'],
                          normalize='index') * 100

print("=== PAÍS vs CLIENTE ALTO VALOR ===\n")
print(tabla_pais.round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
for val, color, label in zip([0,1], ['steelblue','coral'],
                               ['Bajo Valor','Alto Valor']):
    grupo = rfm[rfm['Cliente_Alto_Valor'] == val]['Es_UK']
    axes[0].hist(grupo, bins=3, alpha=0.5,
                 color=color, label=label, edgecolor='black')
axes[0].set_title('País vs Cliente Alto Valor\nHistograma')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Resto del mundo', 'Reino Unido'])
axes[0].legend()

# Barras
tabla_pais.plot(kind='bar',
                color=['steelblue', 'coral'],
                edgecolor='black',
                ax=axes[1])
axes[1].set_title('% Alto/Bajo Valor por País\nBarras')
axes[1].set_xlabel('Es_UK (0=Resto, 1=Reino Unido)')
axes[1].set_ylabel('Porcentaje')
axes[1].set_xticklabels(['Resto del mundo', 'Reino Unido'],
                         rotation=0)
axes[1].legend(['Bajo Valor (0)', 'Alto Valor (1)'])
plt.tight_layout()
plt.show()

CONCLUSIÓN PAÍS:

Los clientes internacionales (fuera de UK) muestran
mayor probabilidad de ser Alto Valor (61.7%) comparado
con los clientes del Reino Unido (49%).

Esto sugiere que los clientes fuera de UK son
probablemente mayoristas o empresas que realizan
pedidos grandes desde el exterior, mientras que
los clientes UK tienen un comportamiento más mixto
entre compras personales y comerciales.

Por tanto Es_UK es una variable útil para el modelo
aunque su poder predictivo es moderado comparado
con Frequency y Productos_Distintos.

2.4 Matríz de correlación

In [ ]:
import seaborn as sns

# Calcular correlación
correlacion = rfm_modelo.corr()

# Visualizar heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlacion,
            annot=True,
            fmt='.2f',
            cmap='coolwarm',
            center=0,
            vmin=-1,
            vmax=1,
            square=True,
            linewidths=0.5)

plt.title('Matriz de Correlación — Features del Modelo',
          fontsize=14)
plt.tight_layout()
plt.show()

CONCLUSIONES MATRIZ DE CORRELACIÓN:

1. Las features más correlacionadas con Cliente_Alto_Valor son:
   → Frequency_log (0.67) y Productos_Distintos_log (0.60)
   → Serán probablemente las más importantes en el modelo

2. No existe multicolinealidad severa entre features
   (ninguna correlación supera 0.8 entre predictores)
   → Todas las variables pueden mantenerse en el modelo

3. Es_UK muestra correlación casi nula (-0.07)
   → Es el predictor más débil del conjunto

4. Recency tiene correlación negativa (-0.38)
   → Confirma que menor Recency = más probable Alto Valor

**NOTA**: A pesar de que Es_UK tiene una correlación prácticamente nula, vamos a mantenerla, ya que quiero capturar todo el contexto de negocio.

Correlación con variable objetivo

In [ ]:
# Correlación específica con variable objetivo
print("=== CORRELACIÓN CON CLIENTE_ALTO_VALOR ===\n")

correlacion_objetivo = correlacion['Cliente_Alto_Valor']\
                       .drop('Cliente_Alto_Valor')\
                       .sort_values(ascending=False)

for feature, valor in correlacion_objetivo.items():
    barra = '█' * int(abs(valor) * 20)
    direccion = '↑' if valor > 0 else '↓'
    print(f"{feature:30} {valor:+.3f} {direccion} {barra}")

# ---

# Ejecuta ambos bloques y comparte los resultados. Lo que buscaremos:
# Correlación con Cliente_Alto_Valor:
# → valor > 0.3   ✅ predictor bueno
# → valor 0.1-0.3 ⚠️ predictor moderado
# → valor < 0.1   ❌ predictor débil

# Correlación entre features:
# → valor > 0.8   ⚠️ redundancia, considerar eliminar una

DESICIÓN: Mantenemos la variable Es_UK

In [ ]:
# Es_UK se mantiene en el modelo
# JUSTIFICACIÓN:
# A pesar de su baja correlación (-0.07) con la variable objetivo,
# Es_UK captura una diferencia de comportamiento real:
# → Clientes internacionales: 61.7% son Alto Valor
# → Clientes UK:              49.0% son Alto Valor
# Su impacto será analizado en Fase III mediante
# importancia de features

features_modelo = ['Recency_log',
                   'Frequency_log',
                   'Ticket_Promedio_log',
                   'Productos_Distintos_log',
                   'Es_UK']

CONCLUSIONES DEL EDA:

1. Variables más predictoras:
   → Frequency_log (0.67) y
     Productos_Distintos_log (0.60)

2. No hay multicolinealidad severa
   → Todas las features se mantienen

3. Es_UK se mantiene por argumento de negocio
   → Clientes internacionales más propensos
     a ser Alto Valor

4. Dataset balanceado 50/50
   → No requiere técnicas de balanceo

5. Hipótesis principal:
   → Un cliente que compra frecuentemente
     y con variedad de productos tiene
     alta probabilidad de ser Alto Valor

3. FASE III.- Modelado

3.1 Preprocesamiento

In [ ]:
# SEPARAR X e Y
# Separar features y variable objetivo
X = rfm_modelo.drop('Cliente_Alto_Valor', axis=1)
y = rfm_modelo['Cliente_Alto_Valor']

print("=== SEPARACIÓN X e Y ===")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFeatures (X):")
print(X.columns.tolist())
print(f"\nVariable objetivo (y):")
print(y.value_counts())

In [ ]:
# DIVIDIR EN DATOS DE ENTRENAMIENTO Y DATOS DE PRUEBA
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% para test
    random_state=42,      # reproducibilidad
    stratify=y            # mantener balance 50/50
)

print("=== TRAIN/TEST SPLIT ===")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\nBalance en train:")
print(y_train.value_counts(normalize=True).round(3) * 100)
print(f"\nBalance en test:")
print(y_test.value_counts(normalize=True).round(3) * 100)

In [ ]:
# ESCALAR
from sklearn.preprocessing import StandardScaler

# Crear el escalador
scaler = StandardScaler()

# Entrenar SOLO con train
X_train_scaled = scaler.fit_transform(X_train)

# Aplicar a test sin reentrenar
X_test_scaled = scaler.transform(X_test)

# Convertir a DataFrame para mejor visualización
X_train_scaled = pd.DataFrame(X_train_scaled,
                               columns=X.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,
                               columns=X.columns)

print("=== ESCALADO ===")
print(f"\nEstadísticas X_train escalado:")
display(X_train_scaled.describe().round(2))

Regresión Logística

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay)

# Entrenar modelo
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)

# Predecir
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

# Métricas
print("=== REGRESIÓN LOGÍSTICA ===\n")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_lr):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob_lr):.4f}")

# Matriz de confusión
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_lr)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Bajo Valor',
                                               'Alto Valor'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
plt.title('Matriz de Confusión — Regresión Logística')
plt.tight_layout()
plt.show()

REGRESIÓN LOGÍSTICA — CONCLUSIÓN:

El modelo obtiene métricas extraordinariamente
altas (F1: 0.99, AUC-ROC: 0.9998), clasificando
correctamente 849 de 857 clientes.

Solo 8 errores:
→ 3 FP: clientes pequeños mal clasificados
→ 5 FN: clientes grandes no detectados

Nota: Los resultados elevados podrían
estar influenciados por la correlación
indirecta entre Ticket_Promedio y la
variable objetivo. Esto se discutirá
en las conclusiones finales.

Árbol de desición

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Entrenar modelo
dt = DecisionTreeClassifier(random_state=42, max_depth=5)
dt.fit(X_train_scaled, y_train)

# Predecir
y_pred_dt = dt.predict(X_test_scaled)
y_prob_dt = dt.predict_proba(X_test_scaled)[:, 1]

# Métricas
print("=== ÁRBOL DE DECISIÓN ===\n")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_dt):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob_dt):.4f}")

# Matriz de confusión
fig, ax = plt.subplots(figsize=(6, 5))
cm_dt = confusion_matrix(y_test, y_pred_dt)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_dt,
                               display_labels=['Bajo Valor',
                                               'Alto Valor'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
plt.title('Matriz de Confusión — Árbol de Decisión')
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar el árbol
plt.figure(figsize=(25, 10))
plot_tree(dt,
          feature_names=X.columns.tolist(),
          class_names=['Bajo Valor', 'Alto Valor'],
          filled=True,
          rounded=True,
          fontsize=10)
plt.title('Árbol de Decisión — Profundidad 3', fontsize=14)
plt.tight_layout()
plt.show()

# Importancia de features
print("=== IMPORTANCIA DE FEATURES ===\n")
importancias_dt = pd.DataFrame({
    'Feature':    X.columns,
    'Importancia': dt.feature_importances_
}).sort_values('Importancia', ascending=False)

display(importancias_dt)

plt.figure(figsize=(8, 5))
plt.barh(importancias_dt['Feature'],
         importancias_dt['Importancia'],
         color='steelblue', edgecolor='black')
plt.title('Importancia de Features — Árbol de Decisión')
plt.xlabel('Importancia')
plt.tight_layout()
plt.show()

ÁRBOL DE DECISIÓN — CONCLUSIÓN:

Supera a la Regresión Logística en todas
las métricas con solo 3 errores de 857.

Hallazgo clave:
→ Frequency_log es el predictor dominante
→ Ticket_Promedio_log complementa la decisión
→ Las demás variables son redundantes

La simplicidad del árbol confirma que
el problema de clasificación tiene una
estructura clara y separable con pocas variables.

Frequency_log domina porque:

1. Es el mejor separador matemático
   entre Alto y Bajo Valor

2. Tiene sentido de negocio:
   clientes frecuentes = clientes valiosos

3. Es componente directo del gasto total:
   Monetary = Frequency × Ticket_Promedio

4. En retail online la frecuencia
   es el mejor indicador de lealtad
   y valor del cliente

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Entrenar modelo
rf = RandomForestClassifier(n_estimators=100,
                             random_state=42)
rf.fit(X_train_scaled, y_train)

# Predecir
y_pred_rf = rf.predict(X_test_scaled)
y_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]

# Métricas
print("=== RANDOM FOREST ===\n")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_rf):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob_rf):.4f}")

# Matriz de confusión
fig, ax = plt.subplots(figsize=(6, 5))
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf,
                               display_labels=['Bajo Valor',
                                               'Alto Valor'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
plt.title('Matriz de Confusión — Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
# Importancia de features
print("=== IMPORTANCIA DE FEATURES — RANDOM FOREST ===\n")
importancias_rf = pd.DataFrame({
    'Feature':     X.columns,
    'Importancia': rf.feature_importances_
}).sort_values('Importancia', ascending=False)

display(importancias_rf)

plt.figure(figsize=(8, 5))
plt.barh(importancias_rf['Feature'],
         importancias_rf['Importancia'],
         color='steelblue', edgecolor='black')
plt.title('Importancia de Features — Random Forest')
plt.xlabel('Importancia')
plt.tight_layout()
plt.show()

RANDOM FOREST — CONCLUSIÓN:

Iguala al Árbol en métricas globales pero
revela una imagen más completa:

→ Frequency_log sigue siendo dominante (43%)
→ Productos_Distintos_log emerge como
  tercer predictor importante (19%)
  que el árbol simple ignoraba
→ Es_UK confirma ser prácticamente
  irrelevante (0.4%)

El Random Forest es más informativo
para entender el comportamiento del cliente
aunque ambos modelos tienen métricas similares.

Modelo SVM

In [ ]:
from sklearn.svm import SVC

# Entrenar modelo
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)

# Predecir
y_pred_svm = svm.predict(X_test_scaled)
y_prob_svm = svm.predict_proba(X_test_scaled)[:, 1]

# Métricas
print("=== SVM (RBF) ===\n")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_svm):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_svm):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_svm):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob_svm):.4f}")

# Matriz de confusión
fig, ax = plt.subplots(figsize=(6, 5))
cm_svm = confusion_matrix(y_test, y_pred_svm)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_svm,
                               display_labels=['Bajo Valor', 'Alto Valor'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
plt.title('Matriz de Confusión — SVM')
plt.tight_layout()
plt.show()

In [ ]:
import shap

explainer = shap.KernelExplainer(svm.predict_proba,
                                  shap.sample(X_train_scaled, 100))
shap_values = explainer.shap_values(X_test_scaled[:100])

# Diagnóstico para ver qué estructura devuelve tu versión de SHAP
print(f"Tipo de shap_values: {type(shap_values)}")
if isinstance(shap_values, list):
    print(f"Número de elementos: {len(shap_values)}")
    for i, sv in enumerate(shap_values):
        print(f"  shap_values[{i}].shape: {sv.shape}")
else:
    print(f"shap_values.shape: {shap_values.shape}")

In [ ]:
# shap_values.shape = (100 muestras, 5 features, 2 clases)
# Clase positiva (Alto Valor) = índice 1 en la última dimensión

shap.summary_plot(shap_values[:, :, 1],
                  X_test_scaled[:100],
                  feature_names=X.columns.tolist(),
                  plot_type='bar')

In [ ]:
shap.summary_plot(shap_values[:, :, 1],
                  X_test_scaled[:100],
                  feature_names=X.columns.tolist())
# plot_type='dot' es el default — muestra si cada feature
# empuja hacia Alto Valor (derecha) o Bajo Valor (izquierda)

AJUSTE DE HIPERPARÁMETROS

In [ ]:
from sklearn.model_selection import GridSearchCV

# ============================================================
# AJUSTE DE HIPERPARÁMETROS — GridSearchCV
# ============================================================

# --- Regresión Logística ---
param_grid_lr = {
    'C':       [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver':  ['liblinear']
}
gs_lr = GridSearchCV(LogisticRegression(random_state=42, max_iter=1000),
                     param_grid_lr, cv=5, scoring='f1', n_jobs=-1)
gs_lr.fit(X_train_scaled, y_train)
print(f"LR   — Mejores params: {gs_lr.best_params_}  |  F1: {gs_lr.best_score_:.4f}")

# --- Random Forest ---
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 5, 10],
    'min_samples_split': [2, 5]
}
gs_rf = GridSearchCV(RandomForestClassifier(random_state=42),
                     param_grid_rf, cv=5, scoring='f1', n_jobs=-1)
gs_rf.fit(X_train_scaled, y_train)
print(f"RF   — Mejores params: {gs_rf.best_params_}  |  F1: {gs_rf.best_score_:.4f}")

# --- SVM ---
param_grid_svm = {
    'C':     [0.1, 1, 10],
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf']
}
gs_svm = GridSearchCV(SVC(probability=True, random_state=42),
                      param_grid_svm, cv=5, scoring='f1', n_jobs=-1)
gs_svm.fit(X_train_scaled, y_train)
print(f"SVM  — Mejores params: {gs_svm.best_params_}  |  F1: {gs_svm.best_score_:.4f}")

In [ ]:
# Evaluar modelos optimizados en test
print("\n=== MODELOS OPTIMIZADOS CON GRIDSEARCHCV ===\n")

modelos_opt = {
    'LR  optimizado': gs_lr.best_estimator_,
    'RF  optimizado': gs_rf.best_estimator_,
    'SVM optimizado': gs_svm.best_estimator_,
}

for nombre, modelo in modelos_opt.items():
    y_pred = modelo.predict(X_test_scaled)
    y_prob = modelo.predict_proba(X_test_scaled)[:, 1]
    print(f"{nombre}:")
    print(f"  F1-Score: {f1_score(y_test, y_pred):.4f}")
    print(f"  AUC-ROC:  {roc_auc_score(y_test, y_prob):.4f}")
    print()

In [ ]:
# Tabla comparativa: antes vs después de GridSearchCV
resultados_grid = []
for nombre, gs in [('Regresión Logística', gs_lr),
                   ('Random Forest',       gs_rf),
                   ('SVM',                 gs_svm)]:
    y_pred = gs.best_estimator_.predict(X_test_scaled)
    y_prob = gs.best_estimator_.predict_proba(X_test_scaled)[:, 1]
    resultados_grid.append({
        'Modelo':          nombre,
        'Mejores Params':  str(gs.best_params_),
        'F1 CV (train)':   round(gs.best_score_, 4),
        'F1 Test':         round(f1_score(y_test, y_pred), 4),
        'AUC-ROC Test':    round(roc_auc_score(y_test, y_prob), 4),
    })

df_grid = pd.DataFrame(resultados_grid)
print("=== RESUMEN GRIDSEARCHCV ===\n")
display(df_grid)

print("""
CONCLUSIÓN GRIDSEARCHCV:
Los hiperparámetros por defecto de scikit-learn ya son
competitivos para este dataset dado su tamaño moderado
(~4,000 clientes) y el buen balance de clases (50/50).
GridSearchCV confirma que la configuración base es cercana
al óptimo, con mejoras marginales en algunos modelos.
""")

Comparación de Modelos

In [ ]:
# === TABLA COMPARATIVA ===
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import pandas as pd

modelos = {
    'Regresión Logística': (y_pred_lr,  y_prob_lr),
    'Árbol de Decisión':   (y_pred_dt,  y_prob_dt),
    'Random Forest':       (y_pred_rf,  y_prob_rf),
    'SVM':                 (y_pred_svm, y_prob_svm),
    # --- Optimizados con GridSearchCV ---
    'LR  (GridSearch)':  (gs_lr.best_estimator_.predict(X_test_scaled),
                          gs_lr.best_estimator_.predict_proba(X_test_scaled)[:,1]),
    'RF  (GridSearch)':  (gs_rf.best_estimator_.predict(X_test_scaled),
                          gs_rf.best_estimator_.predict_proba(X_test_scaled)[:,1]),
    'SVM (GridSearch)':  (gs_svm.best_estimator_.predict(X_test_scaled),
                          gs_svm.best_estimator_.predict_proba(X_test_scaled)[:,1]),
}

resultados = []
for nombre, (y_pred, y_prob) in modelos.items():
    resultados.append({
        'Modelo':     nombre,
        'Accuracy':   accuracy_score(y_test, y_pred),
        'Precision':  precision_score(y_test, y_pred),
        'Recall':     recall_score(y_test, y_pred),
        'F1-Score':   f1_score(y_test, y_pred),
        'AUC-ROC':    roc_auc_score(y_test, y_prob),
    })

df_resultados = pd.DataFrame(resultados).set_index('Modelo')
print("=== COMPARACIÓN DE MODELOS ===\n")
display(df_resultados.round(4))

# Identificar el mejor modelo por F1
mejor_modelo = df_resultados['F1-Score'].idxmax()
print(f"\n✅ Mejor modelo por F1-Score: {mejor_modelo}")
print(f"   F1 = {df_resultados.loc[mejor_modelo, 'F1-Score']:.4f}")

In [ ]:
# === GRÁFICO COMPARATIVO ===
metricas = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
x = range(len(metricas))

# Separar modelos originales vs optimizados para mejor visualización
modelos_originales  = ['Regresión Logística', 'Árbol de Decisión',
                       'Random Forest', 'SVM']
modelos_optimizados = ['LR  (GridSearch)', 'RF  (GridSearch)', 'SVM (GridSearch)']

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Subplot 1: Originales ---
colores_orig = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']
ancho = 0.2
for i, (nombre, color) in enumerate(zip(modelos_originales, colores_orig)):
    valores    = df_resultados.loc[nombre, metricas].values
    posiciones = [pos + i * ancho for pos in x]
    axes[0].bar(posiciones, valores, width=ancho,
                label=nombre, color=color, edgecolor='black', alpha=0.85)

axes[0].set_xticks([pos + ancho * 1.5 for pos in x])
axes[0].set_xticklabels(metricas)
axes[0].set_ylim(0.85, 1.02)
axes[0].set_ylabel('Score')
axes[0].set_title('Modelos Originales', fontsize=13)
axes[0].legend(loc='lower right', fontsize=8)
axes[0].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5)

# --- Subplot 2: Optimizados vs su base ---
pares = [('Regresión Logística', 'LR  (GridSearch)', 'steelblue'),
         ('Random Forest',       'RF  (GridSearch)', 'mediumseagreen'),
         ('SVM',                 'SVM (GridSearch)', 'mediumpurple')]

ancho2 = 0.15
posiciones_base = list(range(len(metricas)))
for i, (orig, opt, color) in enumerate(pares):
    offset_orig = [pos + i * ancho2 * 2 for pos in posiciones_base]
    offset_opt  = [pos + ancho2 for pos in offset_orig]

    vals_orig = df_resultados.loc[orig, metricas].values
    vals_opt  = df_resultados.loc[opt,  metricas].values

    axes[1].bar(offset_orig, vals_orig, width=ancho2,
                color=color, alpha=0.4, edgecolor='black',
                label=f'{orig} (base)')
    axes[1].bar(offset_opt, vals_opt, width=ancho2,
                color=color, alpha=0.9, edgecolor='black',
                label=f'{opt}')

axes[1].set_xticks([pos + ancho2 * 2.5 for pos in posiciones_base])
axes[1].set_xticklabels(metricas)
axes[1].set_ylim(0.85, 1.02)
axes[1].set_ylabel('Score')
axes[1].set_title('Base vs GridSearchCV (por modelo)', fontsize=13)
axes[1].legend(loc='lower right', fontsize=7)
axes[1].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Comparación de Modelos — Originales vs Optimizados',
             fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# === CURVAS ROC ===
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Estilos: originales línea sólida, optimizados línea punteada
config_modelos = {
    'Regresión Logística': ('steelblue',      '-',  2.0),
    'Árbol de Decisión':   ('coral',           '-',  2.0),
    'Random Forest':       ('mediumseagreen',  '-',  2.0),
    'SVM':                 ('mediumpurple',    '-',  2.0),
    'LR  (GridSearch)':    ('steelblue',      '--', 1.5),
    'RF  (GridSearch)':    ('mediumseagreen', '--', 1.5),
    'SVM (GridSearch)':    ('mediumpurple',   '--', 1.5),
}

# --- Subplot 1: Todos los modelos ---
for nombre, (y_pred, y_prob) in modelos.items():
    color, estilo, lw = config_modelos[nombre]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, color=color, linestyle=estilo,
                 lw=lw, label=f'{nombre} (AUC={auc:.4f})')

axes[0].plot([0,1],[0,1], 'k--', lw=1, label='Aleatorio')
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos')
axes[0].set_title('Curvas ROC — Todos los Modelos')
axes[0].legend(loc='lower right', fontsize=7)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.02])

# --- Subplot 2: Solo mejores (RF original vs RF GridSearch) ---
pares_roc = [('Random Forest',     'mediumseagreen'),
             ('RF  (GridSearch)',   'darkgreen'),
             ('SVM',               'mediumpurple'),
             ('SVM (GridSearch)',   'indigo')]

for nombre, color in pares_roc:
    y_prob = modelos[nombre][1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    estilo = '--' if 'GridSearch' in nombre else '-'
    axes[1].plot(fpr, tpr, color=color, linestyle=estilo,
                 lw=2, label=f'{nombre} (AUC={auc:.4f})')

axes[1].plot([0,1],[0,1], 'k--', lw=1, label='Aleatorio')
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].set_title('ROC — Mejores Modelos Base vs GridSearch')
axes[1].legend(loc='lower right', fontsize=8)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.02])

plt.suptitle('Curvas ROC — Comparación Completa', fontsize=14)
plt.tight_layout()
plt.show()

**PCA**

In [ ]:
from sklearn.decomposition import PCA
import time

# ============================================================
# PASO 1: PCA EXPLORATORIO — ¿Cuántos componentes necesitamos?
# ============================================================
pca_full = PCA(random_state=42)
pca_full.fit(X_train_scaled)

varianza_explicada = pca_full.explained_variance_ratio_
varianza_acumulada = np.cumsum(varianza_explicada)

print("=== VARIANZA EXPLICADA POR COMPONENTE ===\n")
for i, (var, acum) in enumerate(zip(varianza_explicada, varianza_acumulada)):
    barra = '█' * int(var * 50)
    print(f"PC{i+1}: {var*100:5.1f}%  acumulada: {acum*100:5.1f}%  {barra}")

In [ ]:
# ============================================================
# PASO 2: SCREE PLOT + VARIANZA ACUMULADA
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree Plot
axes[0].bar(range(1, len(varianza_explicada)+1),
            varianza_explicada * 100,
            color='steelblue', edgecolor='black')
axes[0].plot(range(1, len(varianza_explicada)+1),
             varianza_explicada * 100,
             marker='o', color='red', linewidth=2)
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Varianza Explicada (%)')
axes[0].set_title('Scree Plot')
axes[0].set_xticks(range(1, len(varianza_explicada)+1))

# Varianza Acumulada
axes[1].plot(range(1, len(varianza_acumulada)+1),
             varianza_acumulada * 100,
             marker='o', color='coral', linewidth=2)
axes[1].axhline(y=95, color='red',   linestyle='--', alpha=0.7, label='95%')
axes[1].axhline(y=90, color='orange',linestyle='--', alpha=0.7, label='90%')
axes[1].axhline(y=80, color='green', linestyle='--', alpha=0.7, label='80%')
axes[1].fill_between(range(1, len(varianza_acumulada)+1),
                     varianza_acumulada * 100, alpha=0.1, color='coral')
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('Varianza Acumulada (%)')
axes[1].set_title('Varianza Acumulada — Criterio de Selección')
axes[1].set_xticks(range(1, len(varianza_acumulada)+1))
axes[1].legend()

plt.suptitle('PCA — Selección de Componentes Principales', fontsize=14)
plt.tight_layout()
plt.show()

# Decisión automática
n_90 = np.argmax(varianza_acumulada >= 0.90) + 1
n_95 = np.argmax(varianza_acumulada >= 0.95) + 1
print(f"Componentes para explicar 90% de varianza: {n_90}")
print(f"Componentes para explicar 95% de varianza: {n_95}")

In [ ]:
# ============================================================
# PASO 3: JUSTIFICACIÓN Y DECISIÓN
# ============================================================
print("""
=== JUSTIFICACIÓN DE COMPONENTES ===

Criterios evaluados:
  1. Regla del codo (Scree Plot): punto donde la curva
     se aplana — indica componentes con varianza marginal baja.
  2. Umbral de varianza acumulada: se busca ≥ 90%.
  3. Parsimonia: el menor número de componentes que
     preserve la información suficiente.

Decisión: usar 3 componentes principales.

Justificación:
  → PC1 + PC2 + PC3 explican >90% de la varianza total.
  → Con solo 5 features originales, reducir a 3 componentes
    mantiene la información esencial eliminando redundancia.
  → El scree plot muestra codo claro después de PC3,
    indicando que PC4 y PC5 aportan varianza marginal.
""")

n_componentes = 3  # DECISIÓN FINAL

In [ ]:
# ============================================================
# PASO 4: TRANSFORMAR DATOS CON PCA SELECCIONADO
# ============================================================
pca = PCA(n_components=n_componentes, random_state=42)

X_train_pca = pca.fit_transform(X_train_scaled)   # fit SOLO en train
X_test_pca  = pca.transform(X_test_scaled)         # transform en test

print(f"X_train original: {X_train_scaled.shape}  →  PCA: {X_train_pca.shape}")
print(f"X_test  original: {X_test_scaled.shape}   →  PCA: {X_test_pca.shape}")
print(f"\nVarianza explicada por componente:")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {var*100:.2f}%")
print(f"  Total: {pca.explained_variance_ratio_.sum()*100:.2f}%")

In [ ]:
# ============================================================
# PASO 5: VISUALIZACIÓN 2D Y 3D DE LA SEPARACIÓN
# ============================================================

# Scatter 2D — PC1 vs PC2
plt.figure(figsize=(8, 6))
for clase, color, label in zip([0,1], ['steelblue','coral'],
                                ['Bajo Valor','Alto Valor']):
    mask = y_train == clase
    plt.scatter(X_train_pca[mask, 0], X_train_pca[mask, 1],
                c=color, alpha=0.4, s=15, label=label)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA 2D — Separación de Clases en Espacio Reducido')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PASO 6: LOADINGS — QUÉ FEATURES COMPONEN CADA PC
# ============================================================
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(n_componentes)],
    index=X.columns
)

print("=== LOADINGS (contribución de cada feature a cada PC) ===\n")
display(loadings.round(3))

plt.figure(figsize=(8, 5))
sns.heatmap(loadings, annot=True, fmt='.2f',
            cmap='coolwarm', center=0,
            linewidths=0.5)
plt.title('Loadings PCA — Contribución de Features a Componentes')
plt.tight_layout()
plt.show()

print("""
Interpretación de los componentes:
  PC1: captura principalmente Frequency y Productos_Distintos
       → representa el 'comportamiento de compra activo'
  PC2: captura Recency y Ticket_Promedio
       → representa la 'calidad y recencia de compra'
  PC3: captura Es_UK y varianza residual
       → representa el 'contexto geográfico'
""")

In [ ]:
# ============================================================
# PASO 7: ENTRENAR MEJORES MODELOS CON PCA Y COMPARAR
# ============================================================

resultados_comparacion = []

modelos_a_comparar = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Regresión Logística': LogisticRegression(random_state=42, max_iter=1000),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
}

for nombre, modelo in modelos_a_comparar.items():

    # --- SIN PCA ---
    t0 = time.time()
    modelo.fit(X_train_scaled, y_train)
    t_train_orig = time.time() - t0

    t0 = time.time()
    y_pred_orig = modelo.predict(X_test_scaled)
    t_pred_orig = time.time() - t0
    y_prob_orig = modelo.predict_proba(X_test_scaled)[:, 1]

    resultados_comparacion.append({
        'Modelo':      nombre,
        'Espacio':     'Original (5 features)',
        'F1-Score':    f1_score(y_test, y_pred_orig),
        'AUC-ROC':     roc_auc_score(y_test, y_prob_orig),
        'Accuracy':    accuracy_score(y_test, y_pred_orig),
        'T_entreno(s)': round(t_train_orig, 4),
        'T_pred(s)':    round(t_pred_orig, 4),
    })

    # --- CON PCA ---
    t0 = time.time()
    modelo.fit(X_train_pca, y_train)
    t_train_pca = time.time() - t0

    t0 = time.time()
    y_pred_pca = modelo.predict(X_test_pca)
    t_pred_pca = time.time() - t0
    y_prob_pca = modelo.predict_proba(X_test_pca)[:, 1]

    resultados_comparacion.append({
        'Modelo':      nombre,
        'Espacio':     f'PCA ({n_componentes} componentes)',
        'F1-Score':    f1_score(y_test, y_pred_pca),
        'AUC-ROC':     roc_auc_score(y_test, y_prob_pca),
        'Accuracy':    accuracy_score(y_test, y_pred_pca),
        'T_entreno(s)': round(t_train_pca, 4),
        'T_pred(s)':    round(t_pred_pca, 4),
    })

df_comparacion = pd.DataFrame(resultados_comparacion)
print("=== COMPARACIÓN ORIGINAL vs PCA ===\n")
display(df_comparacion.set_index(['Modelo','Espacio']).round(4))

In [ ]:
# === ENTRENAR MEJORES MODELOS CON PCA ===
# Agrega rf_gs (Random Forest optimizado) además del rf original

rf_gs_pca = gs_rf.best_estimator_.__class__(
    **gs_rf.best_params_, random_state=42)
rf_gs_pca.fit(X_train_pca, y_train)
y_pred_gs_pca = rf_gs_pca.predict(X_test_pca)
y_prob_gs_pca = rf_gs_pca.predict_proba(X_test_pca)[:, 1]

print("=== COMPARACIÓN FINAL: ORIGINAL vs PCA vs GRIDSEARCH+PCA ===\n")
comparacion_final = {
    'RF Original':          (y_pred_rf,     y_prob_rf),
    'RF + GridSearch':      (gs_rf.best_estimator_.predict(X_test_scaled),
                             gs_rf.best_estimator_.predict_proba(X_test_scaled)[:,1]),
    'RF + PCA':             (y_pred_pca,    y_prob_pca),
    'RF + GridSearch + PCA':(y_pred_gs_pca, y_prob_gs_pca),
}

for nombre, (y_pred, y_prob) in comparacion_final.items():
    print(f"{nombre:30}  F1: {f1_score(y_test, y_pred):.4f}  "
          f"AUC: {roc_auc_score(y_test, y_prob):.4f}")

print("""
CONCLUSIÓN:
Esta tabla responde directamente al requisito del enunciado:
'Evalúen y comparen el rendimiento con los modelos entrenados
en el espacio de características original. Discutan el impacto
de PCA en el rendimiento y el tiempo de entrenamiento.'
""")

In [ ]:
# Gráfico comparativo Original vs PCA
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metrica in zip(axes, ['F1-Score', 'AUC-ROC']):
    pivot = df_comparacion.pivot(index='Modelo',
                                  columns='Espacio',
                                  values=metrica)
    pivot.plot(kind='bar', ax=ax, color=['steelblue','coral'],
               edgecolor='black', alpha=0.85)
    ax.set_title(f'{metrica} — Original vs PCA')
    ax.set_ylabel(metrica)
    ax.set_ylim(0.80, 1.02)
    ax.tick_params(axis='x', rotation=15)
    ax.legend(title='Espacio')
    ax.axhline(y=0.90, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Impacto de PCA en el Rendimiento de los Modelos', fontsize=13)
plt.tight_layout()
plt.show()

print("""
=== CONCLUSIÓN PCA ===

Impacto en rendimiento:
  → Random Forest: pérdida mínima de F1 (<0.01)
    PCA preserva la información suficiente para este modelo.
  → Regresión Logística: rendimiento similar o igual
    Las features linealmente transformadas son compatibles.
  → SVM: puede mejorar ligeramente con PCA
    La reducción de ruido beneficia al hiperplano de separación.

Impacto en tiempo de entrenamiento:
  → Con solo 5 features el beneficio en tiempo es marginal
    en este dataset pequeño (~4,000 clientes).
  → En datasets grandes (>100,000 filas, >50 features)
    PCA reduciría drásticamente el tiempo de entrenamiento.

Conclusión: Para este problema específico PCA no mejora
el rendimiento porque las 5 features ya son pocas y
relevantes. El espacio original es preferible por su
interpretabilidad directa.
""")

4. FASE IV — Interpretación y Conclusiones

In [ ]:
# ============================================================
# 4.1 INTERPRETACIÓN DEL MEJOR MODELO — RANDOM FOREST
# ============================================================

print("=== MEJOR MODELO: RANDOM FOREST ===")
print("Justificación de selección:")
print(f"  F1-Score:  {f1_score(y_test, y_pred_rf):.4f}")
print(f"  AUC-ROC:   {roc_auc_score(y_test, y_prob_rf):.4f}")
print("""
  Random Forest fue seleccionado porque:
  1. Mayor F1-Score entre todos los modelos evaluados
  2. Robusto ante outliers (mayoristas con Quantity alta)
  3. No requiere supuestos de distribución de los datos
  4. Provee importancia de features interpretable
  5. Captura relaciones no lineales entre variables
""")

# Importancia de features — Random Forest
importancias_rf_df = pd.DataFrame({
    'Feature':     X.columns,
    'Importancia': rf.feature_importances_
}).sort_values('Importancia', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colores = ['#e63946' if i == 0 else '#457b9d'
           for i in range(len(importancias_rf_df))]
ax.barh(importancias_rf_df['Feature'],
        importancias_rf_df['Importancia'],
        color=colores, edgecolor='black')
ax.set_title('Importancia de Features — Random Forest\n(Mejor Modelo)',
             fontsize=13)
ax.set_xlabel('Importancia (Gini Importance)')
for i, (val, feat) in enumerate(zip(importancias_rf_df['Importancia'],
                                     importancias_rf_df['Feature'])):
    ax.text(val + 0.005, i, f'{val:.3f}', va='center')
plt.tight_layout()
plt.show()

print("\nRanking de importancia:")
for _, row in importancias_rf_df.iterrows():
    barra = '█' * int(row['Importancia'] * 50)
    print(f"  {row['Feature']:30} {row['Importancia']:.3f}  {barra}")

In [ ]:
# ============================================================
# 4.2 SHAP — INTERPRETACIÓN AVANZADA (OPCIONAL AVANZADO)
# ============================================================
import shap

explainer_rf  = shap.TreeExplainer(rf)
shap_values_rf = explainer_rf.shap_values(X_test_scaled)

# Summary plot — beeswarm
plt.figure()
shap.summary_plot(shap_values_rf[:, :, 1],
                  X_test_scaled,
                  feature_names=X.columns.tolist(),
                  show=False)
plt.title('SHAP — Impacto de Features en Predicción (Random Forest)')
plt.tight_layout()
plt.show()

# Bar plot global
plt.figure()
shap.summary_plot(shap_values_rf[:, :, 1],
                  X_test_scaled,
                  feature_names=X.columns.tolist(),
                  plot_type='bar',
                  show=False)
plt.title('SHAP — Importancia Global de Features')
plt.tight_layout()
plt.show()

In [ ]:
# Explicación de 3 clientes individuales
print("=== EXPLICACIÓN DE PREDICCIONES INDIVIDUALES (SHAP) ===\n")

indices_ejemplo = [0, 1, 2]  # 3 primeros clientes del test

for idx in indices_ejemplo:
    pred     = rf.predict(X_test_scaled.iloc[[idx]])[0]
    prob     = rf.predict_proba(X_test_scaled.iloc[[idx]])[0][1]
    etiqueta = '⭐ Alto Valor' if pred == 1 else 'Bajo Valor'

    print(f"Cliente #{idx+1}:")
    print(f"  Predicción:   {etiqueta}")
    print(f"  Probabilidad: {prob:.3f}")
    print(f"  Features:")
    for col in X.columns:
        print(f"    {col}: {X_test_scaled.iloc[idx][col]:.3f}")
    print()

In [ ]:
# ============================================================
# 4.3 CONCLUSIONES Y RECOMENDACIONES DE NEGOCIO
# ============================================================
print("""
╔══════════════════════════════════════════════════════════════════╗
║           FASE IV — CONCLUSIONES Y RECOMENDACIONES              ║
╚══════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RESPUESTA A LA PREGUNTA DE INVESTIGACIÓN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
¿Es posible predecir si un cliente realizará compras de alto valor
basándonos en su comportamiento histórico?

RESPUESTA: Sí, con alta precisión.
El modelo Random Forest alcanza F1-Score de 0.99 y AUC-ROC de 0.99,
clasificando correctamente al 99% de los clientes del conjunto
de prueba usando únicamente variables de comportamiento histórico.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
VARIABLES MÁS IMPORTANTES Y PATRONES ENCONTRADOS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Frequency_log (43%): La frecuencia de compra es el predictor
   dominante. Un cliente que regresa más de 2 veces tiene alta
   probabilidad de ser Alto Valor. Patrón: la lealtad precede
   al valor económico.

2. Ticket_Promedio_log (20%): El gasto por visita es el segundo
   predictor. Clientes que gastan más por compra tienden a
   mantener ese comportamiento en el tiempo.

3. Productos_Distintos_log (19%): La variedad de la canasta
   predice valor. Clientes que exploran más del catálogo
   generan mayor gasto total acumulado.

4. Recency_log (17%): Clientes que compraron recientemente
   tienen mayor probabilidad de ser Alto Valor. La inactividad
   es señal temprana de bajo valor futuro.

5. Es_UK (0.4%): La geografía es prácticamente irrelevante
   una vez que se controla por comportamiento de compra.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RECOMENDACIONES ACCIONABLES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. PROGRAMA DE FIDELIZACIÓN TEMPRANA
   Identificar clientes en su 2da o 3ra compra con ticket
   alto y activar descuentos progresivos para consolidar
   el hábito de compra antes de que alcancen la 5ta compra.
   → Fundamento: Frequency es el predictor #1.

2. CAMPAÑA DE REACTIVACIÓN POR RECENCY
   Segmentar clientes con más de 60 días sin comprar y
   enviar oferta personalizada basada en su historial de
   productos. El modelo identifica estos casos con 99%
   de precisión antes de que se pierdan definitivamente.
   → Fundamento: Recency con correlación negativa de -0.38.

3. ESTRATEGIA DE CROSS-SELLING POR VARIEDAD
   Clientes con Productos_Distintos_log bajo pero
   Frequency alta son candidatos perfectos para
   recomendaciones de nuevos productos del catálogo.
   Aumentar su variedad de canasta aumenta directamente
   su probabilidad de ser Alto Valor.
   → Fundamento: Productos_Distintos es el 3er predictor.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LIMITACIONES DEL ANÁLISIS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. COBERTURA TEMPORAL PARCIAL
   Se analizó únicamente la Hoja 1 del dataset (2009-2010),
   que representa el 38% de las transacciones totales.
   Los patrones de comportamiento de la Hoja 2 (2010-2011)
   podrían diferir, afectando la generalización del modelo.

2. POSIBLE FUGA INDIRECTA DE DATOS (DATA LEAKAGE)
   Ticket_Promedio = Monetary / Frequency. Aunque se excluyó
   Monetary directamente, Ticket_Promedio contiene información
   parcial de Monetary. Un modelo más conservador debería
   excluir también Ticket_Promedio y medir el impacto real
   en las métricas.

3. DEFINICIÓN BINARIA DE VALOR
   La variable objetivo se definió como corte en la mediana,
   lo que fuerza una división 50/50 artificial. En la práctica
   real, la distribución de valor de clientes sigue una ley
   de Pareto (20% de clientes generan 80% del valor), que
   este modelo no captura completamente.

4. CLIENTES MAYORISTAS NO SEPARADOS
   El dataset mezcla clientes retail individuales con
   empresas mayoristas (Quantity > 1,000 unidades).
   Un modelo separado para cada segmento podría ser
   más preciso y accionable.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TRABAJO FUTURO
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Incorporar Hoja 2 (2010-2011) y aplicar validación
   cruzada temporal (TimeSeriesSplit) para simular
   predicción real hacia el futuro.

2. Reformular el problema como predicción de CHURN
   (¿este cliente dejará de comprar en los próximos
   90 días?) — más accionable para el equipo comercial.

3. Explorar modelos de series de tiempo (Prophet, LSTM)
   para predecir el valor futuro del cliente (CLV)
   en lugar de clasificar el valor histórico.

4. Separar el dataset en segmento mayorista vs retail
   y entrenar modelos especializados para cada uno.
""")

In [ ]:
import nbformat

# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
import os

# Buscar el notebook en Drive
result = subprocess.run(['find', '/content/drive', '-name', '*.ipynb'],
                       capture_output=True, text=True)
print(result.stdout)

In [ ]:
import nbformat

ruta = '/content/drive/MyDrive/Colab Notebooks/Proyecto_Final_Aprendizaje_Estadístico_Renato_Pérez.ipynb'

# Leer el notebook
with open(ruta, 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

# Eliminar metadata problemática de widgets
if 'widgets' in nb.metadata:
    del nb.metadata['widgets']

# Guardar limpio en el mismo lugar
with open(ruta, 'w', encoding='utf-8') as f:
    nbformat.write(nb, f)

print("✅ Metadata eliminada correctamente")

In [ ]:
import nbformat
import json

ruta = '/content/drive/MyDrive/Colab Notebooks/Proyecto_Final_Aprendizaje_Estadístico_Renato_Pérez.ipynb'

with open(ruta, 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

# Limpiar metadata global
if 'widgets' in nb.metadata:
    del nb.metadata['widgets']

# Limpiar metadata dentro de cada celda
for cell in nb.cells:
    if 'metadata' in cell:
        cell.metadata = {}
    # Limpiar outputs de cada celda
    if hasattr(cell, 'outputs'):
        for output in cell.outputs:
            if 'metadata' in output:
                output.metadata = {}

with open(ruta, 'w', encoding='utf-8') as f:
    nbformat.write(nb, f)

print("✅ Limpieza completa de todas las celdas")

In [ ]:
import json

ruta = '/content/drive/MyDrive/Colab Notebooks/Proyecto_Final_Aprendizaje_Estadístico_Renato_Pérez.ipynb'

with open(ruta, 'r', encoding='utf-8') as f:
    nb = json.load(f)

# Limpiar metadata global
if 'widgets' in nb.get('metadata', {}):
    del nb['metadata']['widgets']

# Limpiar cada celda
for cell in nb.get('cells', []):
    # Limpiar metadata de celda
    cell['metadata'] = {}
    # Limpiar outputs
    for output in cell.get('outputs', []):
        if 'metadata' in output:
            del output['metadata']
        # Eliminar outputs tipo widget
        if output.get('output_type') == 'display_data':
            if 'application/vnd.jupyter.widget-view+json' in output.get('data', {}):
                del output['data']['application/vnd.jupyter.widget-view+json']

with open(ruta, 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=1)

print("✅ Limpieza JSON completa")